# Error Analysis pada Model Logistic Regression untuk Prediksi Penyakit Jantung

**Portfolio Machine Learning**

Notebook ini melakukan analisis kesalahan (*error analysis*) pada model klasifikasi biner yang memprediksi keberadaan penyakit jantung. 
Kita akan menggunakan **Logistic Regression** dan menyelidiki *false positives* (FP) dan *false negatives* (FN) untuk memahami kelemahan model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
%matplotlib inline

In [ ]:
df = pd.read_csv('../../data/heart_disease_data.csv')
print(f'Shape: {df.shape}')
df.head()

## 1. Eksplorasi Data Awal

Dataset ini memiliki 303 sampel dengan 13 fitur medis dan 1 target biner.
Kita lihat distribusi target dan ringkasan statistik sebelum scaling.

In [ ]:
target_col = df.columns[-1]
print('Distribusi Target:')
print(df[target_col].value_counts())
print()
print(df.describe())

## 2. Preprocessing

Fitur akan di-*scale* menggunakan `StandardScaler` dan data dibagi 80:20 untuk training dan testing.

In [ ]:
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')

## 3. Training Logistic Regression

Kita gunakan `C=0.01` (regularisasi kuat) dan `solver='liblinear'`.

In [ ]:
model = LogisticRegression(C=0.01, solver='liblinear', random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {acc:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

## 4. Error Analysis

Akurasi saja tidak cukup. Kita perlu memahami **di mana** model salah dan **mengapa**.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('Confusion Matrix:')
print(cm)
print(f'\nTrue Negative (TN) : {tn}')
print(f'False Positive (FP): {fp}')
print(f'False Negative (FN): {fn}')
print(f'True Positive (TP) : {tp}')

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Logistic Regression');

### 4.1 False Positive Analysis

**False Positive** terjadi ketika model memprediksi *positif (penyakit jantung)* padahal kenyataannya *tidak sakit*.
FP berbahaya karena menyebabkan pasien sehat mengalami kecemasan dan pemeriksaan lanjutan yang tidak perlu.

Mari kita lihat sampel FP dan cari tahu mengapa model salah.

In [ ]:
feature_names = df.columns[:-1]
test_df = pd.DataFrame(X_test, columns=feature_names)
test_df['actual'] = y_test
test_df['predicted'] = y_pred
test_df['probability'] = y_proba

fp_samples = test_df[(test_df['actual'] == 0) & (test_df['predicted'] == 1)]
fn_samples = test_df[(test_df['actual'] == 1) & (test_df['predicted'] == 0)]

print(f'Jumlah False Positive: {len(fp_samples)}')
print(f'Jumlah False Negative: {len(fn_samples)}')
print()
display(fp_samples.style.set_caption('False Positive Samples'))

#### Interpretasi False Positive

Amati pola pada sampel FP di atas. Beberapa kemungkinan penyebab:
- **Nilai ambang batas (*threshold*)**: Model mungkin terlalu sensitif (threshold 0.5 default). Pasien dengan probabilitas di sekitar 0.5 rentan salah klasifikasi.
- **Overlap fitur**: Pasien sehat yang memiliki nilai fitur mirip dengan pasien sakit (misal: *thalach* rendah, *oldpeak* tinggi, atau *age* tua) cenderung diprediksi positif.
- **Data tidak representatif**: Mungkin subgroup tertentu (misal: wanita usia tertentu) kurang terwakili dalam data training sehingga model tidak belajar dengan baik.

Mari kita bandingkan distribusi fitur antara FP dan prediksi benar untuk mendeteksi pola sistematis.

In [ ]:
tp_samples = test_df[(test_df['actual'] == 1) & (test_df['predicted'] == 1)]
tn_samples = test_df[(test_df['actual'] == 0) & (test_df['predicted'] == 0)]

fp_samples['error_type'] = 'False Positive'
fn_samples['error_type'] = 'False Negative'
tp_samples['error_type'] = 'True Positive'
tn_samples['error_type'] = 'True Negative'

comparison_df = pd.concat([
    fp_samples, fn_samples, tp_samples, tn_samples
], axis=0)

print('DataFrame perbandingan prediksi salah vs benar:')
print(comparison_df[['error_type'] + list(feature_names) + ['probability']].shape)

### 4.2 Visualisasi Distribusi Fitur

Boxplot berikut membandingkan distribusi fitur antara FP, FN, TP, dan TN.
Fitur yang menunjukkan pemisahan jelas antar grup adalah kandidat untuk *feature engineering* lebih lanjut.

In [ ]:
key_features = ['age', 'thalach', 'oldpeak', 'ca', 'chol', 'trestbps']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for i, feat in enumerate(key_features):
    sns.boxplot(data=comparison_df, x='error_type', y=feat, ax=axes[i],
                palette={'False Positive': '#e74c3c', 'False Negative': '#f39c12',
                         'True Positive': '#2ecc71', 'True Negative': '#3498db'})
    axes[i].set_title(f'Distribusi {feat} berdasarkan Tipe Error')
    axes[i].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.suptitle('Perbandingan Distribusi Fitur: FP vs FN vs TP vs TN', y=1.02, fontsize=14);

### 4.3 False Negative Analysis

**False Negative** terjadi ketika model memprediksi *negatif (tidak sakit)* padahal kenyataannya *sakit*.
FN adalah kesalahan paling berbahaya karena pasien yang benar-benar sakit tidak mendapatkan penanganan medis.

Mari kita analisis sampel FN.

In [ ]:
display(fn_samples.style.set_caption('False Negative Samples'))

#### Interpretasi False Negative

Beberapa kemungkinan penyebab FN:
- **Threshold terlalu tinggi**: Model membutuhkan probabilitas > 0.5 untuk memprediksi positif. Pasien sakit dengan probabilitas di bawah 0.5 (misal 0.4-0.49) akan terlewat.
- **Fitur tidak cukup informatif**: Mungkin pasien ini memiliki profil risiko yang tidak tertangkap oleh fitur yang ada (misal: faktor genetik atau gaya hidup).
- **Class imbalance**: Meskipun dataset cukup seimbang, subgroup tertentu dalam kelas positif bisa saja *underrepresented*.
- **Regularisasi terlalu kuat**: Dengan `C=0.01`, model mungkin *underfit* sehingga batas keputusan terlalu sederhana.

### 4.4 Confidence Analysis

Seberapa yakin model saat membuat prediksi yang salah?
Apakah model *sangat yakin* tapi tetap salah? Ini penting untuk mengetahui apakah model memiliki *calibration* yang buruk.

In [ ]:
errors = pd.concat([fp_samples, fn_samples], axis=0)

print('=== Confidence pada Prediksi Salah ===')
print()
print('Deskripsi probabilitas untuk prediksi salah:')
print(errors['probability'].describe())
print()
print('Distribusi confidence:')
print(pd.cut(errors['probability'], bins=[0, 0.3, 0.4, 0.5, 0.6, 0.7, 1.0],
            labels=['0-0.3', '0.3-0.4', '0.4-0.5', '0.5-0.6', '0.6-0.7', '0.7-1.0']).value_counts().sort_index())
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(fp_samples['probability'], bins=10, color='#e74c3c', alpha=0.7, edgecolor='white')
axes[0].axvline(x=0.5, color='black', linestyle='--', label='threshold=0.5')
axes[0].set_title('Distribusi Probabilitas - False Positive')
axes[0].set_xlabel('Predicted Probability (positif)')
axes[0].set_ylabel('Jumlah Sampel')
axes[0].legend()

axes[1].hist(fn_samples['probability'], bins=10, color='#f39c12', alpha=0.7, edgecolor='white')
axes[1].axvline(x=0.5, color='black', linestyle='--', label='threshold=0.5')
axes[1].set_title('Distribusi Probabilitas - False Negative')
axes[1].set_xlabel('Predicted Probability (positif)')
axes[1].set_ylabel('Jumlah Sampel')
axes[1].legend()

plt.tight_layout()
plt.show()
print()

high_conf_errors = errors[errors['probability'] > 0.6]
if len(high_conf_errors) > 0:
    print(f'\nTerdapat {len(high_conf_errors)} prediksi salah dengan confidence > 0.6 (model yakin tapi salah):')
    display(high_conf_errors[['probability', 'error_type'] + list(feature_names)].style.set_caption('High Confidence Errors'))
else:
    print('\nTidak ada prediksi salah dengan confidence > 0.6.')

### 4.5 Parallel Coordinates Plot

Visualisasi *parallel coordinates* membantu melihat pola multidimensi dari sampel yang salah prediksi.

In [ ]:
from pandas.plotting import parallel_coordinates

plot_df = comparison_df[['error_type'] + list(feature_names)].copy()
for col in feature_names:
    plot_df[col] = (plot_df[col] - plot_df[col].min()) / (plot_df[col].max() - plot_df[col].min() + 1e-10)

plt.figure(figsize=(18, 6))
parallel_coordinates(plot_df, class_column='error_type',
                     color=['#e74c3c', '#f39c12', '#2ecc71', '#3498db'],
                     alpha=0.6, linewidth=1.2)
plt.title('Parallel Coordinates - Pola Multi-Fitur dari Tiap Tipe Error', fontsize=13)
plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 4.6 Feature Importance dari Koefisien Model

Melihat koefisien Logistic Regression membantu memahami fitur apa yang paling memengaruhi keputusan model.

In [ ]:
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in coef_df['coefficient']]
plt.barh(coef_df['feature'], coef_df['coefficient'], color=colors)
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.xlabel('Koefisien')
plt.title('Feature Importance - Logistic Regression Coefficients')
plt.tight_layout()
plt.show()

print('Koefisien model:')
print(coef_df.to_string(index=False))

## 5. Insight & Recommendation

Berdasarkan error analysis di atas, berikut ringkasan temuan dan rekomendasi perbaikan:

### Ringkasan Temuan

| Metrik | Nilai |
|--------|-------|
| True Negative (TN) | {tn} |
| False Positive (FP) | {fp} |
| False Negative (FN) | {fn} |
| True Positive (TP) | {tp} |

### Analisis False Positive
- FP terjadi pada pasien yang secara fitur mirip dengan pasien positif (misal: *thalach* rendah, *oldpeak* tinggi).
- Kemungkinan model terlalu sensitif pada fitur tertentu sehingga pasien sehat dengan nilai abnormal pada satu fitur langsung diprediksi positif.

### Analisis False Negative
- FN adalah error paling kritis. Pasien sakit yang terlewatkan bisa berakibat fatal.
- Beberapa sampel FN memiliki probabilitas mendekati threshold (0.4-0.5), menunjukkan bahwa *threshold tuning* bisa membantu.
- Fitur seperti *ca* dan *thal* mungkin perlu encoding yang lebih baik.

### Rekomendasi

1. **Threshold Tuning**:
   - Turunkan threshold klasifikasi (misal dari 0.5 ke 0.3-0.4) untuk mengurangi FN.
   - Ini akan meningkatkan recall (sensitivitas) dengan trade-off menaikkan FP.
   - Untuk kasus medis, recall lebih penting daripada precision.

2. **Feature Engineering**:
   - Buat fitur interaksi antara *age* & *chol*, atau *oldpeak* & *thalach*.
   - Pertimbangkan binning untuk fitur *age*, *trestbps*, dan *chol*.
   - Fitur *ca* (jumlah pembuluh darah) dan *thal* memiliki korelasi tinggi dengan target.

3. **Data Collection**:
   - Tambah data pada subgroup yang sering salah prediksi (terlihat dari boxplot).
   - Jika FP banyak pada usia tertentu, kumpulkan lebih banyak data sehat di rentang usia tersebut.

4. **Model Complexity**:
   - `C=0.01` cukup kuat regularisasinya. Coba tingkatkan `C` (kurangi regularisasi) atau gunakan model non-linear (Random Forest, XGBoost).
   - Logistic Regression adalah model linear — mungkin batas keputusan sebenarnya non-linear.

5. **Cost-sensitive Learning**:
   - Beri bobot lebih pada kelas positif (penyakit jantung) saat training agar model lebih fokus mengurangi FN.
   - Di sklearn: `class_weight='balanced'` atau `class_weight={0:1, 1:2}`.

6. **Calibration**:
   - Jika model sering yakin tapi salah, gunakan *Platt scaling* atau *isotonic regression* untuk kalibrasi probabilitas yang lebih baik.

### Kesimpulan

Error analysis menunjukkan bahwa logistic regression dengan default threshold 0.5 memiliki keterbatasan dalam memisahkan kelas. 
FN dan FP terkonsentrasi pada sampel dengan profil fitur yang *borderline*. Rekomendasi utama adalah **threshold tuning** (menurunkan threshold) 
dan **feature engineering** untuk menangkap pola non-linear. Untuk production, model ensemble seperti Random Forest atau Gradient Boosting 
kemungkinan akan memberikan performa lebih baik karena mampu menangkap interaksi fitur yang kompleks.

In [ ]:
# Rekomendasi threshold tuning - simulasi
thresholds = [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]
results = []
for thresh in thresholds:
    pred_tuned = (y_proba >= thresh).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, pred_tuned).ravel()
    sensitivity = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0
    specificity = tn_t / (tn_t + fp_t) if (tn_t + fp_t) > 0 else 0
    precision = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0
    results.append({
        'Threshold': thresh,
        'Accuracy': accuracy_score(y_test, pred_tuned),
        'Sensitivity (Recall)': sensitivity,
        'Specificity': specificity,
        'Precision': precision,
        'FP': fp_t,
        'FN': fn_t
    })

threshold_df = pd.DataFrame(results)
print('\nSimulasi Threshold Tuning:')
print(threshold_df.to_string(index=False))

---
**Notebook ini adalah bagian dari portfolio Machine Learning.**

*Error analysis adalah langkah kritis dalam siklus ML yang memungkinkan kita bergerak dari sekadar metrik agregat menuju pemahaman mendalam tentang perilaku model.*